# Lab 10 · QA chéo bảng: reviews đối chiếu listings

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 10**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 10 làm sạch bảng `listings`. Lab này kiểm định thứ mà mọi pipeline
nhiều bảng phải kiểm: **hai bảng có kể cùng một câu chuyện không** — khoá ngoại, khoá
tự nhiên, và các cột "tự khai" đối chiếu với con số đếm được.

*Từ tuần này lab ngắn hơn (~50 phút) — 30 phút cuối dành cho BTL clinic (mục cuối notebook).*

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — micro-exercise 🔒
  cuối giờ đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Kiểm tra miền thời gian và khoá ngoại giữa hai bảng — và diễn giải đúng khi phép kiểm "bắt được" thứ không phải lỗi.
2. Chứng minh một bảng **không có khoá tự nhiên** và nói được hệ quả.
3. Đối chiếu cột dẫn xuất "tự khai" với con số tự đếm — xử lý phần lệch một cách chuyên nghiệp.
4. Đóng gói mọi phép kiểm thành `qa_report` ghi ra file.

In [ ]:
import pandas as pd

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
ds = pd.read_csv(f"{BASE}/listings.csv", parse_dates=["last_review"])
rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
len(ds), len(rv)

### Bước 1 · Miền thời gian của reviews

Snapshot mang ngày danh nghĩa **29/06/2026** — review đề ngày sau mốc đó đáng để kiểm.

In [ ]:
# TODO: tìm ngày review sớm nhất, muộn nhất; đếm review CÓ NGÀY SAU 2026-06-29
ngay_dau = ...
ngay_cuoi = ...
so_sau_moc = ...

# --- Ô kiểm tra ---
assert str(ngay_dau)[:10] == "2010-11-13"
assert str(ngay_cuoi)[:10] == "2026-07-01"
assert so_sau_moc == 204
print(f"Review trải từ {ngay_dau:%Y-%m-%d} đến {ngay_cuoi:%Y-%m-%d} — {so_sau_moc} review SAU mốc danh nghĩa!")

204 review đề ngày **sau** 29/06 (muộn nhất 01/07). Lỗi dữ liệu? Không hẳn: "ngày
snapshot" là ngày *danh nghĩa* — trình thu thập của Inside Airbnb chạy trong vài ngày.
Phát hiện đúng, kết luận đúng là: **mốc thời gian thực của dữ liệu là 01/07, không phải
29/06** — pipeline của bạn nên lấy `rv["date"].max()` làm mốc thay vì tin tên thư mục.
(Đây cũng chính là nguồn gốc "49 review tháng 7" bí ẩn của buổi 8.)

### Bước 2 · Khoá ngoại: review "mồ côi"

Mọi `listing_id` trong reviews phải tồn tại trong listings. Nếu không — review đó thuộc về
một phòng ma.

In [ ]:
# TODO: đếm review có listing_id KHÔNG nằm trong ds["id"] (dùng isin và ~)
so_mo_coi = ...

# --- Ô kiểm tra ---
assert so_mo_coi == 0
print("0 review mồ côi — khoá ngoại toàn vẹn.")

Phép kiểm này ra **0 vi phạm** — và đó vẫn là kết quả đáng ghi: "đã kiểm X, không thấy
vi phạm" khác hẳn "chưa kiểm X". Báo cáo QA của bài tập lớn ghi cả những dòng 0.

### Bước 3 · Khoá tự nhiên: bảng reviews rút gọn có khoá không?

In [ ]:
# TODO: đếm số dòng trùng theo cặp (listing_id, date) — duplicated với subset
so_trung_cap = ...

# --- Ô kiểm tra ---
assert so_trung_cap == 3548
print(f"{so_trung_cap:,} dòng 'trùng' theo (listing_id, date).")

3.548 dòng "trùng" — nhưng **không phải lỗi**: hai khách khác nhau hoàn toàn có thể
review cùng một phòng trong cùng một ngày. Kết luận đúng là: bản rút gọn này **không có
khoá tự nhiên** (bản đầy đủ có cột `id` riêng cho từng review). Hệ quả thực tế: không được
dùng `drop_duplicates` trên bảng này — sẽ xoá oan 3.548 review thật. Trùng hay không
phụ thuộc **khoá bạn định nghĩa** — đúng bài buổi 10.

### Bước 4 · Cột "tự khai" đối chiếu con số tự đếm

`listings` mang sẵn `number_of_reviews` và `number_of_reviews_ltm` (12 tháng gần nhất) —
do Inside Airbnb tính hộ. Tin được không? **Tự đếm mà đối chiếu.**

In [ ]:
# TODO: đếm số review THẬT của từng listing từ bảng rv (groupby listing_id + size),
#       đưa về ds theo id (map hoặc reindex), điền 0 cho listing không có review
dem_that = ...
ds["dem_that"] = ...

# TODO: đếm số listing có number_of_reviews KHÁC dem_that
so_lech_tong = ...

# --- Ô kiểm tra ---
assert so_lech_tong == 0
print("18.534/18.534 listing: số tự khai khớp đếm thật 100%. Hai bảng cùng một nguồn sinh.")

In [ ]:
# Giờ đến cột LTM ("last twelve months"). Lấy mốc 12 tháng = từ 2025-06-30:
ltm_that = rv[rv["date"] >= "2025-06-30"].groupby("listing_id").size()
ds["ltm_that"] = ds["id"].map(ltm_that).fillna(0).astype(int)

# TODO: đếm số listing lệch giữa number_of_reviews_ltm và ltm_that,
#       và xem phân bố độ lệch (tự khai − tự đếm) bằng describe
so_lech_ltm = ...
do_lech = ...

# --- Ô kiểm tra ---
assert so_lech_ltm == 665
print(f"{so_lech_ltm} listing lệch LTM; độ lệch trung vị: {do_lech.median():.0f}")
do_lech[do_lech != 0].describe().round(2)

665 listing lệch, hầu hết lệch **−1** (tự khai ít hơn ta đếm đúng 1 review). Thủ phạm
gần như chắc chắn: **định nghĩa mốc "12 tháng"** — Inside Airbnb cắt tại một ngày hơi khác
mốc 30/06/2025 ta chọn. Bài học kép: (1) cột dẫn xuất luôn phụ thuộc *định nghĩa* — khi
dùng phải hoặc tra định nghĩa gốc, hoặc **tự tính bằng định nghĩa của mình và ghi rõ**;
(2) lệch nhỏ có cấu trúc (toàn −1) khác hẳn lệch lung tung — nhìn *phân bố* độ lệch
trước khi kết tội dữ liệu.

### Bước 5 · Đóng gói qa_report

In [ ]:
# TODO: gom 5 phép kiểm thành qa_report (DataFrame 3 cột: quy_tac, so_dong, ghi_chu)
#       rồi ghi ra file qa_report_reviews.csv
qa_report = pd.DataFrame([
    {"quy_tac": "review_sau_moc",      "so_dong": so_sau_moc,    "ghi_chu": "date > ngày danh nghĩa; mốc thực = 01/07"},
    {"quy_tac": "review_mo_coi",      "so_dong": so_mo_coi,     "ghi_chu": "listing_id không có trong listings"},
    {"quy_tac": "trung_theo_cap",     "so_dong": so_trung_cap,  "ghi_chu": "KHÔNG xoá — bảng không có khoá tự nhiên"},
    {"quy_tac": "lech_tong_review",   "so_dong": so_lech_tong,  "ghi_chu": "tự khai vs tự đếm"},
    {"quy_tac": "lech_ltm",           "so_dong": so_lech_ltm,   "ghi_chu": "khác định nghĩa mốc 12 tháng"},
])
# TODO: ghi ra CSV (không index) và đọc lại để kiểm
...

# --- Ô kiểm tra ---
kq = pd.read_csv("qa_report_reviews.csv")
assert kq.shape == (5, 3) and kq["so_dong"].sum() == 204 + 3548 + 665
qa_report

## Bài tự làm 🔓

**Bộ QA du hành thời gian.** Chạy lại đúng 5 phép kiểm trên **snapshot cũ 2025-09-27**
(đổi `BASE` thành `.../2025-09-27/visualisations`, mốc "tương lai" và mốc LTM đổi theo
ngày snapshot đó). Quy tắc nào cho kết quả khác? Đây chính là phép thử "bộ QA sống sót
qua snapshot khác" mà bài tập lớn sẽ chấm bằng snapshot held-out.

In [ ]:
# Viết bài tự làm của bạn ở đây

---

## 🧭 BTL clinic tuần 10 (~30 phút — làm việc theo nhóm, trợ giảng đi từng bàn)

Tự soát theo checklist; đánh dấu ✅/❌ và ghi một dòng "việc tuần tới" cho mỗi ❌:

1. ☐ Repo GitHub **private** đã mời tài khoản GV/TA (danh sách trên Canvas Portal);
   cấu trúc thư mục theo đề (data/ src/ notebooks/ reports/ figures/).
2. ☐ **Script tải dữ liệu** chạy được: tải đủ các snapshot bắt buộc của thành phố chính
   (và thành phố đối chứng) vào `data/raw/` — chưa commit dữ liệu thô lên repo.
3. ☐ Phản hồi của giảng viên cho **bản đề xuất tuần 8** đã được cả nhóm đọc; các mục
   phải sửa đã thành issue/việc được giao.
4. ☐ **Bộ quy tắc QA nháp** (theo buổi 10 + lab này): mỗi quy tắc đủ 4 phần
   tên–điều kiện–lý do–hành động; chạy thử trên snapshot mới nhất, có qa_report đầu tiên.
5. ☐ Phân công tuần tới: ai phụ trách **hợp phần LLM** (buổi 11) — người đó tạo API key
   Gemini trước ở nhà (aistudio.google.com, miễn phí).

> Nhóm xong sớm: thử chạy bộ QA của nhóm trên **một snapshot nhóm chưa dùng** — đúng phép
> thử held-out mà lab hôm nay vừa tập.

## Tóm tắt buổi lab

| Bạn đã làm | Dùng cho |
|---|---|
| Kiểm miền (mốc thực ≠ mốc danh nghĩa) + khoá ngoại | qa_report bài tập lớn |
| Chứng minh bảng không có khoá tự nhiên | quyết định KHÔNG drop_duplicates mù quáng |
| Đối chiếu cột tự khai vs tự đếm, đọc phân bố lệch | thái độ với mọi cột dẫn xuất |
| qa_report.csv | deliverable chuẩn của đề |

Buổi lý thuyết tới: **LLM cho dữ liệu phi cấu trúc** — mang API key Gemini đến lớp.